# Tara N1 Pretraining

- **Data set:** 500M token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **data.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`
- **tara_n1_pretrain_v2.pth:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth`
## Completed training phases:
- Completed Shard 00 - resulting 1B token corpus
- Completed Shard 01 - resulting 2B token corpus
- Completed Shard 02 - resulting 3B token corpus
- Completed Shard 03 - resulting 4B token corpus
- Completed Shard 04 - resulting 5B token corpus

In [1]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")
with open("_data.py", "w") as f:
    f.write(model_res.text)
print("Downloaded data.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

# model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth")
# with open("tara_n1_pretrain.pth", "wb") as f:
#     f.write(model_res.content)
# print("Downloaded wieghts")

Downloaded model.py successfully.
Downloaded data.py successfully.
Downloaded train_utils.py successfully.


In [3]:
from model import *
from train_utils import *
from _data import *

In [4]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=16,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# The Dataset

In [6]:
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download, login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

repo_id = "ShanmukhVashtav/Fineweb-edu-5B-gpt-2-tokenized"
shard_id = 0

shard_path = hf_hub_download(
    repo_id = repo_id,
    filename = f"train/shard_{shard_id:02d}.npy",
    repo_type = "dataset"
)


print(f"Downloaded Shard {shard_id:02d} at {shard_path}")

# tokens = np.load(shard_path, mmap_mode="r")



# print(f"Collected {len(tokens)} tokens.")


Downloaded Shard 00 at /root/.cache/huggingface/hub/datasets--ShanmukhVashtav--Fineweb-edu-5B-gpt-2-tokenized/snapshots/203c3af2d52694a49a1ebbd86cb8a430b50daf78/train/shard_00.npy


In [9]:
train_split = 0.9
train_dataloader, test_dataloader = create_shards_dataloaders(shard_path, train_split, block_size = config.block_size, batch_size=config.batch_size)

Total samples: 999998976
Train samples: 899999078
Test samples: 99999898


# Pretraining the model

In [10]:
# modelV1 = CustomGPT(config)

# if torch.cuda.device_count() > 1:
#     modelV1 = nn.DataParallel(modelV1)
#     print(f"Using {torch.cuda.device_count()} GPUs")

# modelV1.to(device)

# calc_params(modelV1)

# loss_fn = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(modelV1.parameters(), lr=1e-4)
# scaler = torch.amp.GradScaler()


modelV2 = CustomGPT(config)
# state_dict = torch.load("tara_n1_pretrain.pth", map_location=device)
# state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

# modelV2.load_weights("tara_n1_pretrain.pth")

if torch.cuda.device_count() > 1:
    modelV2 = nn.DataParallel(modelV2)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV2.to(device)

calc_params(modelV2)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Using 2 GPUs
Total Parameters: 30,783,057
Trainable Parameters: 30,783,057


In [ ]:
from tqdm.auto import tqdm
steps = 20000
train_iter = iter(train_dataloader)
for step in tqdm(range(1, steps+1)):
    # modelV1.train()
    modelV2.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    # train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    if not train_iter:
        train_iter = iter(train_dataloader)
    train_loss = train_step(modelV2, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    
    if step % 5000 == 0:
        # modelV1.eval()
        modelV2.eval()
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        # test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        test_loss = test_step(modelV2, test_dataloader, 20, loss_fn, device)
        print(f"Step {step} | Train Loss: {train_loss} | Test Loss: {test_loss:}")

  0%|          | 0/20000 [00:00<?, ?it/s]

In [ ]:
torch.save(modelV2.state_dict(), "tara_n1_pretrain_v2_00.pth")
torch.save(optimizer.state_dict(), )

# Testing



In [ ]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

# modelV1.eval()
test_model = CustomGPT(config)
test_model.load_weights("tara_n1_pretrain_v2_00.pth")
# test_model.load_weights("tara_n1_pretrain.pth")
test_model.to(device)
test_model.eval()
with torch.inference_mode():
    # output = modelV1.generate(context, max_new_tokens=100)
    output = test_model.generate(context)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")
